In [52]:
# --- 1. SETUP & IMPORTS ---
import sys
import os
from pathlib import Path
import pandas as pd

# Ajouter le dossier racine au path pour importer 'src'
root_path = Path(os.getcwd()).parent
sys.path.append(str(root_path))

from src.database import get_engine, inspect_table_columns
from notebooks.AttritionCleaner import AttritionCleaner

# --- 2. CONNEXION ---
engine = get_engine()


# ⚙️ CONFIGURATION GLOBALE DU PROJET

In [53]:
RAW_DATA_DIR     = Path("../data/raw")
PREPROCESED_DIR  = Path("../data/processed")
MODELS_DIR       = Path("../models/V2")
FEATURES_DIR     = Path("../src/features")

FEATURE_TARGET           = "attrition_binary"

FEATURE_TARGET_INITIAL   = "a_quitte_l_entreprise"

# 1 : ❇️ Lecture des 3 Fichiers bruts

In [54]:
df_sirh_raw    = pd.read_csv(RAW_DATA_DIR / "extrait_sirh.csv")
df_evals_raw   = pd.read_csv(RAW_DATA_DIR / "extrait_eval.csv")
df_sondage_raw = pd.read_csv(RAW_DATA_DIR / "extrait_sondage.csv")

# 2 :🛢 Creation des 3 Tables

In [55]:
cleaner = AttritionCleaner(engine)

# Paso 1: SIRH
cleaner.create_raw_sirh()
inspect_table_columns("raw_sirh", engine)

# Paso 2: EVALS
cleaner.create_raw_evals()
inspect_table_columns("raw_evals", engine)

# Paso 3: SONDAGE
cleaner.create_raw_sondage()
inspect_table_columns("raw_sondage", engine)

✅ Tabla 'raw_sirh' creada.
✅ Estructura de la tabla: raw_sirh
✅ Tabla 'raw_evals' creada.
✅ Estructura de la tabla: raw_evals
✅ Tabla 'raw_sondage' creada.
✅ Estructura de la tabla: raw_sondage


,column_name,data_type,is_nullable
0,a_quitte_l_entreprise,text,YES
1,nombre_participation_pee,bigint,YES
2,nb_formations_suivies,bigint,YES
3,nombre_employee_sous_responsabilite,bigint,YES
4,code_sondage,bigint,YES
5,distance_domicile_travail,bigint,YES
6,niveau_education,bigint,YES
7,domaine_etude,text,YES
8,ayant_enfants,text,YES
9,frequence_deplacement,text,YES


# 3 : 📝➜🛢️ Ingestion vers PostgreSQL

## NETTOYAGE DES IDENTIFIANTS 

In [56]:
# --- ÉTAPE DE NETTOYAGE DES IDENTIFIANTS ---

# A. SIRH : On s'assure que l'ID est une chaîne de caractères propre
df_sirh_raw['emp_id']    = df_sirh_raw['id_employee'].astype(str).str.strip()

# B. EVAL : On enlève le préfixe 'E_' pour n'avoir que le chiffre
df_evals_raw['emp_id']   = df_evals_raw['eval_number'].astype(str).str.replace('E_', '', regex=False).str.strip()

# C. SONDAGE : On enlève les zéros à gauche (ex: 00001 -> 1)
df_sondage_raw['emp_id'] = df_sondage_raw['code_sondage'].astype(str).str.lstrip('0').str.strip()

print("✅ IDs harmonisés : Les 3 tables utilisent désormais le même format pour 'emp_id'.")

✅ IDs harmonisés : Les 3 tables utilisent désormais le même format pour 'emp_id'.


In [57]:
print("🚀 Iniciando Ingestión...")
cleaner.ingest_sirh(df_sirh_raw)
cleaner.ingest_evals(df_evals_raw)
cleaner.ingest_sondage(df_sondage_raw)

🚀 Iniciando Ingestión...
✅ Tabla 'raw_sirh' cargada y dependencias limpiadas.
✅ Tabla 'raw_evals' cargada y dependencias limpiadas.
✅ Tabla 'raw_sondage' cargada y dependencias limpiadas.


# 4 : 🛢️👀  Creation des Vues avec des donnes propres. 

In [58]:
cleaner.clean_sirh()
cleaner.clean_evals()
cleaner.clean_sondage()


✅ Vista 'v_clean_sirh' creada (SIRH)
✅ Vista 'v_clean_evals' creada (Evaluaciones)
✅ Vista 'v_clean_sondage' creada con éxito


# 5 : 🛢️👀 MERGE. Création de la Vue "v_master_clean"

In [59]:
# 1. Crear la Vista Maestra en SQL
cleaner.create_master_view()

# 2. Traer los datos a Python para el EDA
df_final = cleaner.get_master_data()

# 3. Inspección de calidad final
from src.database import inspect_table_data
inspect_table_data("v_master_clean", engine)

print(f"✅ Dataset merged: {df_final.shape[0]} empleados y {df_final.shape[1]} variables.")

✅ Vista Maestra 'v_master_clean' creada con éxito.
👀 Primeros 5 registros de: v_master_clean


,emp_id,id_employee,age,genre,statut_marital,revenu_mensuel,departement,poste,nombre_experiences_precedentes,annee_experience_totale,...,nombre_participation_pee,nb_formations_suivies,nombre_employee_sous_responsabilite,distance_domicile_travail,nivel_educacion,campo_estudios,tiene_hijos,frecuencia_viajes,annees_depuis_la_derniere_promotion,annes_sous_responsable_actuel
0,1,1,41,F,Célibataire,5993,Commercial,Cadre Commercial,8,8,...,0,0,1,1,2,Infra & Cloud,0,Occasionnel,0,5
1,2,2,49,M,Marié(e),5130,Consulting,Assistant de Direction,1,10,...,1,3,1,8,1,Infra & Cloud,0,Frequent,1,7
2,4,4,37,M,Célibataire,2090,Consulting,Consultant,6,7,...,0,3,1,2,2,Autre,0,Occasionnel,0,0
3,5,5,33,F,Marié(e),2909,Consulting,Assistant de Direction,1,8,...,0,3,1,3,4,Infra & Cloud,0,Frequent,3,0
4,7,7,27,M,Marié(e),3468,Consulting,Consultant,9,6,...,1,3,1,2,1,Transformation Digitale,0,Occasionnel,2,2


✅ Dataset merged: 1470 empleados y 32 variables.


In [60]:
inspect_table_columns("v_master_clean", engine)

✅ Estructura de la tabla: v_master_clean


,column_name,data_type,is_nullable
0,emp_id,character varying,YES
1,id_employee,bigint,YES
2,age,bigint,YES
3,genre,text,YES
4,statut_marital,text,YES
5,revenu_mensuel,bigint,YES
6,departement,text,YES
7,poste,text,YES
8,nombre_experiences_precedentes,bigint,YES
9,annee_experience_totale,bigint,YES


# 5 : 🛢️👀 CLEAN. Création de la Vue "v_master_clean"

In [61]:
cleaner.create_features_view()
# 4. Carga final para EDA y Modelado
df_ml = cleaner.get_features_data()

# Verificamos las nuevas columnas 'feX_'
print(df_ml.filter(like='fe').columns.tolist())
display(df_ml.head())

✅ Vista de Ingeniería de Características 'v_features_engineering' creada.
['fe1_ratio_stagnation', 'fe2_stabilite_manager', 'fe3_indice_job_hopping', 'fe4_anciennete_relative', 'fe5_satisfaction_globale', 'fe6_risque_overwork', 'fe7_penibilite_trajet', 'fe8_valeur_experience']


,emp_id,id_employee,age,genre,statut_marital,revenu_mensuel,departement,poste,nombre_experiences_precedentes,annee_experience_totale,...,annees_depuis_la_derniere_promotion,annes_sous_responsable_actuel,fe1_ratio_stagnation,fe2_stabilite_manager,fe3_indice_job_hopping,fe4_anciennete_relative,fe5_satisfaction_globale,fe6_risque_overwork,fe7_penibilite_trajet,fe8_valeur_experience
0,1,1,41,F,Célibataire,5993,Commercial,Cadre Commercial,8,8,...,0,5,0.0000,1.0000,0.8889,0.2609,2.00,0.5000,1,665.89
1,10,10,59,F,Marié(e),2670,Consulting,Consultant,4,12,...,0,0,0.0000,0.0000,2.4000,0.0244,1.75,0.3333,3,205.38
2,100,100,35,M,Célibataire,4312,Commercial,Cadre Commercial,0,16,...,2,8,0.1250,0.5714,16.0000,0.8824,2.25,0.0000,0,253.65
3,1001,1001,27,F,Marié(e),2811,Consulting,Consultant,9,4,...,2,2,0.6667,0.6667,0.4000,0.2222,2.50,0.0000,0,562.20
4,1002,1002,45,M,Marié(e),3633,Consulting,Consultant,1,9,...,0,8,0.0000,0.8889,4.5000,0.3333,2.75,0.2500,23,363.30


In [62]:
inspect_table_columns('v_features_engineering', engine)

✅ Estructura de la tabla: v_features_engineering


,column_name,data_type,is_nullable
0,emp_id,character varying,YES
1,id_employee,bigint,YES
2,age,bigint,YES
3,genre,text,YES
4,statut_marital,text,YES
5,revenu_mensuel,bigint,YES
6,departement,text,YES
7,poste,text,YES
8,nombre_experiences_precedentes,bigint,YES
9,annee_experience_totale,bigint,YES
